# GTZAN Unsupervised CNN Autoencoder

Reconstruction-only autoencoder on GTZAN mel spectrograms (no genre labels during training).

For the **multi-task** variant (reconstruction + genre loss), see `gtzan_autoencoder.ipynb`.
For a **supervised CNN baseline**, see `gtzan_cnn_baseline.ipynb`.

**Pipeline:**
1. Convert `GTZAN_Data/genres_original/<genre>/*.wav` into cached mel spectrogram `.npy` files.
2. Train a CNN autoencoder on normalized mel spectrograms (MSE only).
3. Evaluate the latent vectors with a linear probe, k-NN, silhouette, PCA, and UMAP.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.neighbors import KNeighborsClassifier

try:
    import umap
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap

# Works when the notebook is run from src/ or from the project root.
NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_CWD.parent if (NOTEBOOK_CWD.parent / 'GTZAN_Data').exists() else NOTEBOOK_CWD
GTZAN_ROOT = PROJECT_ROOT / 'GTZAN_Data'
AUDIO_DIR = GTZAN_ROOT / 'genres_original'
MEL_DIR = GTZAN_ROOT / 'melspecs'
MANIFEST_CSV = GTZAN_ROOT / 'features_30_sec.csv'
MEL_MANIFEST_CSV = GTZAN_ROOT / 'gtzan_mel_manifest.csv'
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 12000
N_FFT = 512
HOP_LENGTH = 256
N_MELS = 96
CROP_LENGTH = 1366
BATCH_SIZE = 16
RANDOM_STATE = 42

print(f'Project root: {PROJECT_ROOT}')
print(f'GTZAN root:   {GTZAN_ROOT}')
print(f'Audio dir:    {AUDIO_DIR}')
print(f'Mel cache:    {MEL_DIR}')
print(f'Manifest:     {MANIFEST_CSV}')

if not AUDIO_DIR.exists():
    raise FileNotFoundError(f'Expected GTZAN WAVs at {AUDIO_DIR}')
if not MANIFEST_CSV.exists():
    raise FileNotFoundError(f'Expected GTZAN manifest at {MANIFEST_CSV}')

In [ ]:
def crop_or_pad_mel(mel, crop_length=CROP_LENGTH):
    if mel.shape[1] > crop_length:
        start = (mel.shape[1] - crop_length) // 2
        mel = mel[:, start:start + crop_length]
    elif mel.shape[1] < crop_length:
        mel = np.pad(mel, ((0, 0), (0, crop_length - mel.shape[1])), mode='constant', constant_values=mel.min())
    return mel.astype(np.float32)


def wav_path_for_row(row):
    return AUDIO_DIR / row['label'] / row['filename']


def mel_path_for_row(row):
    return MEL_DIR / row['label'] / f"{Path(row['filename']).stem}.npy"


def wav_to_mel(wav_path):
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
    mel_power = librosa.feature.melspectrogram(
        y=y,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel_power, ref=np.max)
    return crop_or_pad_mel(mel_db)


def build_mel_cache(force=False):
    df = pd.read_csv(MANIFEST_CSV)
    rows = []
    bad_files = []

    MEL_DIR.mkdir(parents=True, exist_ok=True)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building mel cache'):
        wav_path = wav_path_for_row(row)
        mel_path = mel_path_for_row(row)
        mel_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            if force or not mel_path.exists():
                mel = wav_to_mel(wav_path)
                np.save(mel_path, mel)
            rows.append({
                'filename': row['filename'],
                'label': row['label'],
                'wav_path': str(wav_path),
                'mel_path': str(mel_path),
            })
        except Exception as exc:
            bad_files.append((str(wav_path), str(exc)))

    manifest_df = pd.DataFrame(rows)
    manifest_df.to_csv(MEL_MANIFEST_CSV, index=False)

    print(f'Cached/verified mels: {len(manifest_df)}')
    if bad_files:
        print(f'Skipped unreadable files: {len(bad_files)}')
        for path, err in bad_files[:5]:
            print(f'  {path}: {err}')
    print(f'Mel manifest saved to: {MEL_MANIFEST_CSV}')
    return manifest_df


manifest_df = build_mel_cache(force=False)
display(manifest_df.head())

In [ ]:
genres = sorted(manifest_df['label'].unique())
genre_to_idx = {genre: i for i, genre in enumerate(genres)}
idx_to_genre = {i: genre for genre, i in genre_to_idx.items()}

train_val_df, test_df = train_test_split(
    manifest_df,
    test_size=0.20,
    stratify=manifest_df['label'],
    random_state=RANDOM_STATE,
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.20,
    stratify=train_val_df['label'],
    random_state=RANDOM_STATE,
)

print(f'Genres: {genres}')
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Train class counts:')
print(train_df['label'].value_counts().sort_index())


def compute_mel_stats(df):
    total = 0
    sum_x = 0.0
    sum_x2 = 0.0
    for mel_path in tqdm(df['mel_path'], desc='Computing train mel stats'):
        mel = np.load(mel_path).astype(np.float64)
        total += mel.size
        sum_x += mel.sum()
        sum_x2 += np.square(mel).sum()
    mean = sum_x / total
    var = max(sum_x2 / total - mean ** 2, 1e-8)
    return float(mean), float(np.sqrt(var))


mel_mean, mel_std = compute_mel_stats(train_df)
print(f'Train mel mean: {mel_mean:.4f} | std: {mel_std:.4f}')


class GTZANDataset(Dataset):
    def __init__(self, df, genre_to_idx, mel_mean, mel_std):
        self.df = df.reset_index(drop=True)
        self.genre_to_idx = genre_to_idx
        self.mel_mean = mel_mean
        self.mel_std = mel_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel = np.load(row['mel_path']).astype(np.float32)
        mel = (mel - self.mel_mean) / self.mel_std
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.genre_to_idx[row['label']], dtype=torch.long)
        return mel, label


def make_loaders(batch_size=BATCH_SIZE):
    train_ds = GTZANDataset(train_df, genre_to_idx, mel_mean, mel_std)
    val_ds = GTZANDataset(val_df, genre_to_idx, mel_mean, mel_std)
    test_ds = GTZANDataset(test_df, genre_to_idx, mel_mean, mel_std)

    loader_kwargs = dict(batch_size=batch_size, num_workers=0, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders()
mels, labels = next(iter(train_loader))
print(f'Batch mels: {mels.shape} | labels: {labels.shape}')

In [ ]:
class GenreAutoencoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 4)),
        )

        self.flatten_dim = 512 * 12 * 21
        self.fc_enc = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flatten_dim)

        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=(2, 4)),
            nn.Conv2d(128, 1, kernel_size=3, padding=1),
        )

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.fc_enc(x)

    def decode(self, z):
        x = self.fc_dec(z)
        x = x.view(x.size(0), 512, 12, 21)
        x = self.decoder(x)
        if x.shape[-1] < CROP_LENGTH:
            x = nn.functional.pad(x, (0, CROP_LENGTH - x.shape[-1]))
        return x[:, :, :N_MELS, :CROP_LENGTH]

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


_probe = GenreAutoencoder(latent_dim=128)
_probe_x = torch.randn(2, 1, N_MELS, CROP_LENGTH)
_probe_recon, _probe_z = _probe(_probe_x)
print(f'Input:  {_probe_x.shape}')
print(f'Recon:  {_probe_recon.shape}')
print(f'Latent: {_probe_z.shape}')


In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {device}')

LATENT_DIM = 128
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
BEST_MODEL_PATH = CHECKPOINT_DIR / 'gtzan_unsupervised_ae_best.pth'

model = GenreAutoencoder(latent_dim=LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

best_val_loss = float('inf')
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for mels, _ in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS} train'):
        mels = mels.to(device)
        optimizer.zero_grad()
        recon, _ = model(mels)
        loss = criterion(recon, mels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for mels, _ in val_loader:
            mels = mels.to(device)
            recon, _ = model(mels)
            loss = criterion(recon, mels)
            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss})

    print(f'Epoch {epoch + 1}/{NUM_EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  → saved {BEST_MODEL_PATH} (val loss {val_loss:.4f})')

history_df = pd.DataFrame(history)
display(history_df.tail())

plt.figure(figsize=(8, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('GTZAN Unsupervised Autoencoder Training')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize reconstruction quality on validation examples
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()

mels, labels = next(iter(val_loader))
mels = mels.to(device)
with torch.no_grad():
    recon, _ = model(mels)

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for i in range(4):
    orig = (mels[i, 0].cpu().numpy() * mel_std) + mel_mean
    rec = (recon[i, 0].cpu().numpy() * mel_std) + mel_mean
    genre = idx_to_genre[int(labels[i])]
    for ax, data, title in zip(axes[:, i], [orig, rec], [f'Original: {genre}', 'Reconstruction']):
        img = librosa.display.specshow(data, x_axis='time', y_axis='mel', sr=SAMPLE_RATE, hop_length=HOP_LENGTH, ax=ax)
        ax.set_title(title)
        fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def collect_embeddings(loader):
    model.eval()
    embeddings = []
    labels = []
    for mels, batch_labels in tqdm(loader, desc='Extracting embeddings'):
        z = model.encode(mels.to(device))
        embeddings.append(z.cpu().numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(embeddings, axis=0), np.concatenate(labels, axis=0)


train_embeddings, train_labels = collect_embeddings(train_loader)
val_embeddings, val_labels = collect_embeddings(val_loader)
test_embeddings, test_labels = collect_embeddings(test_loader)

print(f'Train embeddings: {train_embeddings.shape}')
print(f'Val embeddings:   {val_embeddings.shape}')
print(f'Test embeddings:  {test_embeddings.shape}')

In [ ]:
# Quantitative latent-space checks (labels used for evaluation only)
probe = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
probe.fit(train_embeddings, train_labels)
probe_pred = probe.predict(test_embeddings)
probe_acc = accuracy_score(test_labels, probe_pred)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_embeddings, train_labels)
knn_pred = knn.predict(test_embeddings)
knn_acc = accuracy_score(test_labels, knn_pred)

all_embeddings = np.concatenate([train_embeddings, val_embeddings, test_embeddings], axis=0)
all_labels = np.concatenate([train_labels, val_labels, test_labels], axis=0)
sil_score = silhouette_score(all_embeddings, all_labels)

metrics_df = pd.DataFrame([
    {'metric': 'linear_probe_test_accuracy', 'value': probe_acc},
    {'metric': 'knn_5_test_accuracy', 'value': knn_acc},
    {'metric': 'latent_silhouette_score', 'value': sil_score},
    {'metric': 'best_val_reconstruction_mse', 'value': best_val_loss},
])
display(metrics_df)

print('Linear probe classification report:')
print(classification_report(test_labels, probe_pred, target_names=genres))


In [ ]:
# PCA / UMAP latent visualizations colored by genre
def plot_embedding_2d(points, labels, title):
    plt.figure(figsize=(10, 7))
    for genre_idx, genre in idx_to_genre.items():
        mask = labels == genre_idx
        plt.scatter(points[mask, 0], points[mask, 1], s=14, alpha=0.7, label=genre)
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_points = pca.fit_transform(all_embeddings)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum() * 100:.1f}%')
plot_embedding_2d(pca_points, all_labels, 'PCA of Unsupervised AE Latents by Genre')

reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1, random_state=RANDOM_STATE)
umap_points = reducer.fit_transform(all_embeddings)
plot_embedding_2d(umap_points, all_labels, 'UMAP of Unsupervised AE Latents by Genre')
